In [ ]:
from nerfstudio.utils.eval_utils import eval_setup
from nerfstudio.cameras.camera_paths import get_interpolated_camera_path
from pathlib import Path
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import torch
import ddpg
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
downscale = 32

config_path = Path(f'nerfstudio/lego/nerfacto/2024-05-02_094606/config-down-x{downscale}.yml')
db_path = f'nerfstudio/db-x{downscale}.pkl'

_, pipeline, _, _ = eval_setup(config_path, test_mode='val')

train_dataset = pipeline.datamanager.train_dataset
eval_dataset = pipeline.datamanager.eval_dataset
model = pipeline.model

In [ ]:
def ns_next(cameras, indices, debug=False):
    cameras = cameras[torch.tensor(indices)]
    rb = cameras.generate_rays(torch.tensor([[i] for i in range(len(indices))]))
    # flatten the RayBundle
    imshape = rb.shape[:2]
    batch_size = rb.shape[-1]
    rb = rb.reshape((-1, batch_size))
    rb.origins = rb.origins.transpose(0, 1)
    rb.directions = rb.directions.transpose(0, 1)
    rb.camera_indices = rb.camera_indices.transpose(0, 1)
    rb.metadata['directions_norm'] = rb.metadata['directions_norm'].transpose(0, 1)
    rb.pixel_area = rb.pixel_area.transpose(0, 1)
    rb._shape = rb.directions.shape[:-1]
    rb = rb.flatten().to(device)
    rb.imshape = imshape
    rb.imsize = torch.tensor(imshape).prod().item()
    rb.batch_size = batch_size
    rb.chunk_size = 1250
    rb.device = rb.origins.device
    # keep camera parameters for mapping world to image coordinates
    rb.camera_to_world = cameras.camera_to_worlds.to(device)
    rb.camera_intrinsics = cameras.get_intrinsics_matrices().to(device)
    # convert directions to camera coordinates
    rb.directions = ddpg.geom.world_to_camera(rb.origins + rb.directions, rb.camera_to_world)
    rb.directions /= rb.directions.norm(dim=-1, keepdim=True)
    # angle between ray and bottom-right neighboring ray
    rb.clip_angle = torch.acos(
        torch.tensor(
            [(rb.directions[0] @ rb.directions[imshape[1] + 1]).clip(-1, 1)],
            device=device,
        )
    )

    if debug:
        size_bytes = rb.origins.element_size() * rb.origins.nelement() \
            + rb.directions.element_size() * rb.directions.nelement() \
            + rb.camera_indices.element_size() * rb.camera_indices.nelement() \
            + rb.metadata['directions_norm'].element_size() * rb.metadata['directions_norm'].nelement() \
            + rb.pixel_area.element_size() * rb.pixel_area.nelement() \
            + rb.fov_axis.element_size() * rb.fov_axis.nelement() \
            + rb.fov.element_size() * rb.fov.nelement()
        print('camera indices:', indices, 'size (bytes):', size_bytes)

    return rb

In [ ]:
# pre-render NeRF
def render(loader, database, plot=False):
    for rb in loader:
        directions = rb.directions.clone()
        new_directions = ddpg.geom.camera_to_world(rb.directions, rb.camera_to_world) - rb.origins
        rb.directions = new_directions / new_directions.norm(dim=-1, keepdim=True)
        model_outputs = model(rb)
        rb.directions = directions
        rgb = model_outputs['rgb'].detach()
        acc = model_outputs['accumulation'].detach()

        for i in range(0, rb.imsize*rb.batch_size, rb.imsize):
            origin = tuple(rb.origins[i].cpu().tolist())
            rgb_ = rgb[i:i+rb.imsize]
            acc_ = acc[i:i+rb.imsize]
            assert rb.imsize == rgb_.shape[0]
            assert rb.imsize == acc_.shape[0]
            database[origin] = {
                'rgb': rgb_,
                'accumulation': acc_,
                'shape': rb.imshape,
            }
            if plot:
                plt.figure()
                img = rgb_.reshape(*rb.imshape, rgb.size(-1)).cpu().numpy()
                plt.imshow(img)


try:
    db = torch.load(db_path)
except:
    train_loader = ddpg.CustomDL(ns_next, train_dataset.cameras, step_size=1)
    test_loader = ddpg.CustomDL(ns_next, eval_dataset.cameras, step_size=1)

    db = dict()
    render(train_loader, db)
    render(test_loader, db)
    torch.save(db, db_path)

In [ ]:
class CustomNeRF:
    def __init__(self, db):
        self.db = db

    def __call__(self, rb):
        out = dict()
        rgb_ = torch.tensor([], device=rb.device)
        # TODO: accumulation
        for i in range(0, rb.imsize*rb.batch_size, rb.imsize):
            # retrieve image from db
            origin = tuple(rb.origins[i].cpu().tolist())
            img = self.db[origin]
            # select portion of image according to rb shape
            rgb = img['rgb'].reshape(*img['shape'], -1)
            rgb = rgb[:rb.imshape[0], :rb.imshape[1]]
            # find intersection of ray directions with image plane
            points = rb.origins[i:i+rb.imsize] + rb.directions[i:i+rb.imsize]
            img_coords = ddpg.geom.world_to_image(
                points,
                rb.camera_to_world[i//rb.imsize:i//rb.imsize + 1],
                rb.camera_intrinsics[i//rb.imsize:i//rb.imsize + 1],
            ).reshape(*rb.imshape, -1)
            # compute rgb values by bilinear interpolation
            upper_left = img_coords.round().int() - 1 # upper-left of 4 nearest neighbors
            local = img_coords - (upper_left + 0.5) # local coordinates
            k = upper_left[..., 0]
            l = upper_left[..., 1]
            u = local[..., 0].unsqueeze(-1)
            v = local[..., 1].unsqueeze(-1)
            top_left = rgb[k % rb.imshape[0], l % rb.imshape[1]] # RGB values
            bottom_left = rgb[(k+1) % rb.imshape[0], l % rb.imshape[1]]
            top_right = rgb[k % rb.imshape[0], (l+1) % rb.imshape[1]]
            bottom_right = rgb[(k+1) % rb.imshape[0], (l+1) % rb.imshape[1]]
            _rgb_ = (
                (1-v)*((1-u)*top_left + u*bottom_left) + v*((1-u)*top_right + u*bottom_right)
            ).reshape(rb.imsize, -1)
            assert not _rgb_.isnan().any()
            # concatenate results
            rgb_ = torch.cat((rgb_, _rgb_))
        out['rgb'] = rgb_
        return out


custom_nerf = CustomNeRF(db)

In [ ]:
try:
    dist_mat = torch.load('nerfstudio/dist_mat.pkl')
except:
    dl1 = ddpg.CustomDL(ns_next, train_dataset.cameras)
    dl2 = ddpg.CustomDL(ns_next, train_dataset.cameras)

    dist_mat = torch.full((len(dl1), len(dl2)), torch.inf)

    for i, rb1 in enumerate(dl1):
        oi = rb1.origins[0]
        dl2.reset(dl1.step)

        for j, rb2 in enumerate(dl2, start=i+1):
            display(f'{i} {j}')
            clear_output(wait=True)
            oj = rb2.origins[0]

            dist = (oi - oj).norm().item()
            dist_mat[i, j] = dist

    torch.save(dist_mat, 'nerfstudio/dist_mat.pkl')

In [ ]:
# sort camera pairs by ascending distance
k = 6800

dist_flat = dist_mat.flatten()
top_val, idx_flat = dist_flat.topk(k, largest=False)

row_idx = idx_flat // dist_mat.size(1)
col_idx = idx_flat % dist_mat.size(1)
top_idx = torch.stack((row_idx, col_idx), dim=1)

In [ ]:
top_idx, top_val

In [ ]:
# experiment 1: far vs close camera pair
idx = 43 # 0.06

In [ ]:
idx = 155 # 0.12

In [ ]:
idx = 560 # 0.24

In [ ]:
idx = 1995 # 0.48

In [ ]:
idx = 6786 # 0.96

In [ ]:
top_idx[idx], top_val[idx]

In [ ]:
camera_pair = train_dataset.cameras[top_idx[idx]]
camera_interpol = get_interpolated_camera_path(camera_pair, steps=3, order_poses=True)[1:2]

train_loader = ddpg.CustomDL(ns_next, camera_pair, step_size=len(camera_pair))
test_loader = ddpg.CustomDL(ns_next, camera_interpol, step_size=len(camera_interpol))

In [ ]:
config = ddpg.Config(
    warmup=10000,
    mem_batch_size=64,
    max_episodes=False,
    eval_episodes=1,
)
env = ddpg.NeRFStudioEnv(
    custom_nerf,
    reward_thres=.75,
    reward_scale=10,
    # with_radiance=True,
)

In [ ]:
runner = ddpg.Runner(
    train_loader,
    test_loader,
    env,
    config,
    tensorboard=True,
    with_img=True,
    # plot_rays=True,
    # num_rays=10,
)

In [ ]:
runner(mode='train')

In [ ]:
runner.env.model = model

In [ ]:
runner(mode='test', load_weights={
    'actor': f'{runner.save_path}/actor.pkl',
    'critic': f'{runner.save_path}/critic.pkl'
})

In [ ]:
del pipeline, model, runner
gc.collect()
torch.cuda.empty_cache()

In [ ]:
plot_loader = ddpg.CustomDL(ns_next, camera_pair, step_size=2)
plot_model = custom_nerf
fname = 'camera-pose'

In [ ]:
plot_loader = ddpg.CustomDL(ns_next, camera_interpol, step_size=1)
plot_model = model
fname = 'interpol'

In [ ]:
for rb in plot_loader:
    directions = rb.directions.clone()
    rb.directions = ddpg.geom.camera_to_world(rb.directions, rb.camera_to_world) - rb.origins
    out = plot_model(rb)
    rb.directions = directions
    rgb = out['rgb']
    gs = rgb.mean(dim=-1)
    img = gs.reshape(rgb.shape[0] // rb.imsize, *rb.imshape).detach().cpu().numpy()
    ddpg.utils.show_img(img, title=fname, stdout=True, write_path=runner.save_path)

In [ ]:
# experiment 2: camera with 3 neighbors (at different distances)
idx = torch.where(top_idx[:, 0] == top_idx[3, 0])[0]
idx_lst = list(set(top_idx[idx].flatten().tolist()))
idx_lst.remove(top_idx[3, 0].item())
train_idx = torch.tensor(idx_lst)
test_idx = top_idx[3, 0].unsqueeze(-1)

camera_train = train_dataset.cameras[train_idx]
camera_test = train_dataset.cameras[test_idx]

top_idx[idx], top_val[idx], train_idx, test_idx

In [ ]:
train_loader = ddpg.CustomDL(ns_next, camera_train, step_size=len(train_idx))
test_loader = ddpg.CustomDL(ns_next, camera_test, step_size=len(test_idx))

In [ ]:
plot_loader = ddpg.CustomDL(ns_next, camera_train, step_size=len(train_idx))

In [ ]:
plot_loader = ddpg.CustomDL(ns_next, camera_test, step_size=len(test_idx))

In [ ]:
for rb in plot_loader:
    directions = rb.directions.clone()
    rb.directions = ddpg.geom.camera_to_world(rb.directions, rb.camera_to_world) - rb.origins
    out = custom_nerf(rb)
    ref = model(rb)
    rb.directions = directions
    rgb1 = out['rgb']
    gs1 = rgb1.mean(dim=-1)
    img1 = gs1.reshape(rgb1.shape[0] // rb.imsize, *rb.imshape).detach().cpu().numpy()
    ddpg.utils.show_img(img1, title='bilinear interpolation', stdout=True)
    rgb2 = ref['rgb']
    gs2 = rgb2.mean(dim=-1)
    img2 = gs2.reshape(rgb2.shape[0] // rb.imsize, *rb.imshape).detach().cpu().numpy()
    ddpg.utils.show_img(img2, title='NeRF', stdout=True)

In [ ]:
# entire dataset
train_loader = ddpg.CustomDL(ns_next, train_dataset.cameras, step_size=10)
test_loader = ddpg.CustomDL(ns_next, eval_dataset.cameras, step_size=10)